# RD01-c — Observable regimes in signed-return **asymmetry**

**Primary horizon: 30 minutes.** RD01-b's residual-location regime did **not** pass the frozen gate. This notebook tests a distinct possibility: differences in the **left/right shape** of future signed-return densities that cannot be explained merely by changing conditional mean or common variance.

**Registered controls:** symmetric Student-t; interaction-Ridge location + symmetric Student-t; common globally asymmetric Student-t; smooth continuous covariate-dependent asymmetric Student-t; CART state-conditional asymmetric Student-t. The transformed two-piece Student-t has **exactly the same mean and variance** across skew values. Student-t degrees of freedom are fixed at 6. Read the immutable protocol before interpreting any market results.

**Caution:** The normalizer is RD01's causal trailing 24h realized-variance proxy, **NOT the actual AM-T-HAR-Blend point-in-time forecast**. RD01-c is historical discovery only, not confirmation.

In [ ]:
import os,sys,json,time,gc,subprocess,tempfile
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Open this notebook from the repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import observable_regime_discovery as rd
from data_engineering.modeling import observable_regime_discovery_c as rc
SYNTHETIC=os.getenv('RD01C_SYNTHETIC','0')=='1'
cfg=(rc.Spec(horizons=(30,),tree_min_leaf_absolute=25,minimum_state_events=15,
             minimum_state_days=3,bootstrap_repetitions=30,minimum_bootstrap_days=5) if SYNTHETIC else rc.Spec())
if SYNTHETIC:
    temp=tempfile.TemporaryDirectory(prefix='rd01c-')
    OUT=Path(temp.name)/'01c'
else:
    rootval=os.getenv('DERIBIT_DATA_ROOT')
    if not rootval:raise RuntimeError('DERIBIT_DATA_ROOT is not set in .env/environment')
    STORE=Path(rootval)/'research'/'cs04a_feature_store'
    OUT=ROOT/'notebooks'/'Regime_Discovery'/'01c'
OUT.mkdir(parents=True,exist_ok=True)
print('RD01-c',rc.VERSION,'synthetic=',SYNTHETIC,'horizons=',cfg.horizons,'out=',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
test=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_observable_regime_discovery_c.py')],
                    cwd=str(ROOT),env=env,capture_output=True,text=True)
print(test.stdout[-1500:]);print(test.stderr[-650:]);assert test.returncode==0,'RD01-c regression tests failed'


In [ ]:
datasets={};excluded={}
if SYNTHETIC:
    sys.path.insert(0,str(ROOT/'tests'))
    from test_observable_regime_discovery import synthetic_panel
    datasets={'BTC':synthetic_panel(days=94)}
else:
    # The CS04-A local store already contains current-bar momentum, futures flow,
    # basis and intraminute dynamics; 60m future labels are rebuilt causally.
    cols=list(dict.fromkeys(['state_time','return_5m','feature_ready','log_rv24h',
                              *rd.BASE_COLS,*rd.TREE_COLS,'ret_5m','ret_15m','ret_30m']))
    for asset in cfg.assets:
        path=STORE/f'{asset}_features_and_forward_magnitude.parquet'
        if not path.is_file():excluded[asset]=f'Missing local feature store: {path}';continue
        try:
            datasets[asset]=pd.read_parquet(path,columns=cols)
            print(asset,'loaded',len(datasets[asset]),'rows')
        except Exception as e:excluded[asset]=f'{type(e).__name__}: {e}'
assert datasets,f'No market feature stores: {excluded}'
if not SYNTHETIC and not {'BTC','ETH'}.issubset(datasets):
    raise RuntimeError(f'BTC and ETH primary coverage required: {excluded}')


In [ ]:
results={};coverage=[]
for asset,panel in datasets.items():
    t0=time.perf_counter()
    try:
        result=rc.scan_asset(panel,cfg,asset)
        results[asset]=result
        coverage.append({'asset':asset,'complete_days':result['full_days'],
                         'model_score_rows':len(result['scores']),
                         'calibrated_state_rows':len(result['calibration']),
                         'elapsed_seconds':round(time.perf_counter()-t0,2)})
        print(asset,'completed',len(result['scores']),'scores across',cfg.horizons,'horizons')
    except Exception as e:
        excluded[asset]=f'{type(e).__name__}: {e}'
        raise
    finally:
        gc.collect()
assert results
coverage=pd.DataFrame(coverage)
display(coverage)


In [ ]:
out=rc.consolidate(results,cfg,synthetic=SYNTHETIC)
out['manifest']['excluded']=excluded
print('Average equal-asset-fold scores; negative delta is better than direct symmetric t:')
display(out['summary'].sort_values(['horizon_min','mean_nll']))
print('Primary 30m NLL comparisons for regime-dependent shape:')
f=out['bootstrap'].query("horizon_min==30 and metric=='nll' and model=='direct_regime_skew'")
display(f[['asset','fold','reference','delta','ci025','ci975']].head(60))
print('Supported candidate skew states (calibration only):')
display(out['calibration'].query("horizon_min==30").head(25))
print('Decision (discovery only):',json.dumps(out['decision'],indent=2))


In [ ]:
aggregate={
 '01c_scores.csv':'scores','01c_paired_day_bootstrap.csv':'bootstrap',
 '01c_comparison_summary.csv':'summary','01c_state_diagnostics.csv':'states',
 '01c_state_calibration.csv':'calibration','01c_tree_definitions.csv':'definitions',
 '01c_episode_summary.csv':'episodes','01c_transition_matrix.csv':'transitions',
 '01c_exit_hazards.csv':'hazards','01c_persistence.csv':'persistence',
}
for name,key in aggregate.items():out[key].to_csv(OUT/name,index=False)
coverage.to_csv(OUT/'01c_coverage.csv',index=False)
(OUT/'01c_manifest.json').write_text(json.dumps(out['manifest'],indent=2,default=str),encoding='utf-8')
(OUT/'01c_decision.json').write_text(json.dumps(out['decision'],indent=2,default=str),encoding='utf-8')
assert all((OUT/name).is_file() for name in [*aggregate,'01c_coverage.csv','01c_manifest.json','01c_decision.json'])
print('Wrote',len(aggregate)+3,'aggregate outputs. SYNTHETIC=',SYNTHETIC)
print('Do not publish CS04-A local per-origin parquet or synthetic smoke results.')


## Scientific interpretation rules

- A regime improves signed-return distribution only if `direct_regime_skew` beats **continuous skew**, **global skew** and **interaction-symmetric** on the same origins at the frozen primary 30-minute horizon.
- State longevity is descriptive; trees may create persistent labels even under non-regime-generating data. No sticky smoothing is applied.
- The comparison is exploratory after RD01/b/CS04-A historical inspection. A winning bootstrap confidence interval is **not** fresh confirmation.
- All five density models use the same RD01 trailing-variance normalizer; these results **do not** test a real AM-T-HAR-Blend overlay.
- If this module also fails, the right action is to retain direct continuous return predictors and move to truly new data or information sources, not tune another tree on the same windows.